In [5]:
using DataFrames
#using Ipopt
#using JuMP
using VegaLite
using LinearAlgebra

# Simulated Data example (1 sector)

To solve equations 6 and 7, we simply need data on GDP and trade shares. 

Generate Fake Trade Data (used Gemini to generate)

3 Countries

In [6]:
# 1. Define total expenditure (X) for 3 countries
# Units are arbitrary (e.g., billions of dollars)
X = [1000.0, 1500.0, 500.0]

# 2. Define the Trade Share Matrix (pi_matrix)
# pi_matrix[n, i] is the fraction of country n's spending that goes to country i.
# Rows MUST sum to 1.
# Diagonal elements should be the largest to simulate realistic "home bias".
pi_matrix = [
    0.70  0.20  0.10;  # Country 1 spends 70% on itself, 20% on 2, 10% on 3
    0.15  0.80  0.05;  # Country 2 spends 15% on 1, 80% on itself, 5% on 3
    0.25  0.15  0.60   # Country 3 spends 25% on 1, 15% on 2, 60% on itself
]

# Sanity check: Ensure rows sum to exactly 1.0
@assert all(isapprox.(sum(pi_matrix, dims=2), 1.0))

# 3. Calculate GDP (Y) implied by market clearing.
# Country i's GDP is the sum of what every country n spends on country i.
# Y_i = sum_n (pi_{ni} * X_n)
# In matrix math, this is Y = pi_matrix^T * X (transpose of pi multiplied by X)
Y = pi_matrix' * X

# 4. Calculate initial trade deficits (D).
# Deficit is Spending (X) minus Income/GDP (Y). This is the current accounts deficit
D = X - Y 

# 5. Output the results to plug into your Hat Algebra solver
println("--- Simulated Initial Equilibrium ---")
for i in 1:3
    println("Country $i:")
    println("  Spending (X) : ", X[i])
    println("  GDP (Y)      : ", Y[i])
    println("  Deficit (D)  : ", D[i])
    println("  Trade Shares : ", pi_matrix[i, :])
    println()
end

println("Sum of Deficits (Should be 0.0): ", sum(D))

--- Simulated Initial Equilibrium ---
Country 1:
  Spending (X) : 1000.0
  GDP (Y)      : 1050.0
  Deficit (D)  : -50.0
  Trade Shares : [0.7, 0.2, 0.1]

Country 2:
  Spending (X) : 1500.0
  GDP (Y)      : 1475.0
  Deficit (D)  : 25.0
  Trade Shares : [0.15, 0.8, 0.05]

Country 3:
  Spending (X) : 500.0
  GDP (Y)      : 475.0
  Deficit (D)  : 25.0
  Trade Shares : [0.25, 0.15, 0.6]

Sum of Deficits (Should be 0.0): 0.0


In [7]:
X

3-element Vector{Float64}:
 1000.0
 1500.0
  500.0

In [8]:
Y

3-element Vector{Float64}:
 1050.0
 1475.0
  475.0

In [9]:
#Create trade flow matrix 
X_matrix = zeros(Float64, 3,3)
for n in 1:3
    for i in 1:3
        X_matrix[n,i] = pi_matrix[n,i]*X[n]
    end
end

X_matrix

3×3 Matrix{Float64}:
 700.0   200.0  100.0
 225.0  1200.0   75.0
 125.0    75.0  300.0

In [10]:
#Calculate Exports and imports
flows_only = copy(X_matrix)
for i in 1:3
    flows_only[i, i] = 0.0
end

exports = sum(flows_only, dims=1)[:] # Sum of columns: what others buy from i
imports = sum(flows_only, dims=2)[:] # Sum of rows: what n buys from others
println(exports)
println(imports)

#current account surplus for each country. exports - imports
CA = exports - imports
println(CA)

[350.0, 275.0, 175.0]
[300.0, 300.0, 200.0]
[50.0, -25.0, -25.0]


Solving equations 6 and 7 with the counterfactual that:

$D_n^{M'} = D_n^M + CA_n= 0$ 
meaning manufacturing deficit + current accounts deficit = 0 

and $D_n^{'} = 0$

NOTE: In the example that Gemini gave, there is only 1 sector which is manufacturing. So both are equal 


Define Parameters and Exogenous Variable

In [11]:
D_Mprime = imports - exports + CA #trade deficit - CA = 0. the trade deficit is just the opposite of CA
D_prime = zeros(3)

#parameters taken from the paper
θ = 8.28
α = 0.188 
β = 0.312

0.312

## Fixed Point Iteration Solver

In [12]:
#Iteration parameters 
tol = 1e-6
max_iter = 10000
step_size = 0.1

#Initial guesses for endogenous variables w_hat and p_hat
w_hat = ones(3)
p_hat = ones(3)

3-element Vector{Float64}:
 1.0
 1.0
 1.0

In [13]:
iteration_count = 0
error = 1.

N = 3

while error > tol && iteration_count < max_iter

    #Inner Loop: Solve for prices (equation 7)
    p_error = 1
    while p_error > tol
        p_hat_update = zeros(N) #update guess of p_hat by filling in equation 7
        #need to calculate p_n for each N 
        for n in 1:N
            #Calculate term inside parantheses in equation 7
            # term = 0. 
            # for k in 1:N
            #     term += pi_matrix[n,k] * w_hat[k]^(-θ*β) * p_hat[k]^(-θ*(1-β))
            # end
            term = sum([pi_matrix[n,k] * w_hat[k]^(-θ*β) * p_hat[k]^(-θ*(1-β)) for k in 1:N])
            p_hat_update[n] = term^(-1/θ)
        end
       
        p_error = maximum(abs.(p_hat_update - p_hat))
        p_hat = p_hat_update #norm(p_hat_update - p_hat) apparently this is better condition to guarantee convergence
    end

    #In the Outer Loop, solve Equation 6 
    #Equation 6 is a system of N equations. 1 equation for each country 
    LHS = zeros(N) #LHS is income/supply.
    RHS = zeros(N) #RHS is demand 

    for i in 1:N 
        #Calculate LHS 
        LHS[i] = w_hat[i]*Y[i] + D_prime[i] - (1/α)*D_Mprime[i] 

        #Calculate RHS
        sum_RHS = 0 
        for n in 1:N
            numerator = pi_matrix[n,i] * w_hat[i]^(-θ*β) * p_hat[i]^(-θ*(1-β))
            denominator = p_hat[n]^(-θ) #note the denominator is just an expression in terms of p_hat which we already have
            multiplicative_term = w_hat[n]*Y[n] + D_prime[n] - ((1-β)/α)*D_Mprime[n] 

            sum_RHS += (numerator/denominator)*multiplicative_term
        end
        RHS[i] = sum_RHS

    end

    #Now update wage. 
    w_hat_update = zeros(N) #update guess of w_hat 
    for i in 1:N
        #RHS (demand) > LHS (supply/income), increase wages so that LHS increases
        w_hat_update[i] = w_hat[i] * ((RHS[i]/LHS[i])^step_size)
    end

    #Need to keep GDP (Y) constant 
    current_Y = sum(w_hat_update .* Y)
    initial_Y = sum(Y)
    norm_factor = initial_Y/current_Y

    w_hat = w_hat_update*norm_factor

    #check for convergence
    error = maximum(abs.((RHS .- LHS) ./ LHS)) #norm(RHS - LHS) 
    iteration_count += 1
end

println("Convergence achieved in ", iteration_count, " iterations.")
println("Final w_hat:", w_hat)
println("Final p_hat:", p_hat)

Convergence achieved in 25 iterations.
Final w_hat:[1.0069106527067788, 0.9972173545427718, 0.9933646667521975]
Final p_hat:[1.0017706507488224, 0.9992334288009166, 0.9989067882736723]


## NLsolve Solver

https://docs.sciml.ai/NonlinearSolve/stable/api/nlsolve/
https://github.com/JuliaNLSolvers/NLsolve.jl

In [14]:
using NLsolve

NLsolve works by solving F(x) = 0, where x and 0 are vectors. 

Need to reformulate system of equations in the form F(x) = 0. 

x = [wage, prices]

This is not straightfoward. Skipping for now

## Welfare Calculations

In [18]:
Y_prime = w_hat .* Y

welfare_hat = [
    (w_hat[i]/p_hat[i])^α * (1+D_prime[i]/Y_prime[i])/(1+D[i]/Y[i]) for i in 1:N
    ]

println("Counterfactual change in welfare is ", welfare_hat)

Counterfactual change in welfare is [1.0510107396043795, 0.9829600365715823, 0.9490068542373417]


# Simulated Data example (2 sectors)

Manufacturing and non-manufacturing goods

Using Gemini to generate the simulated data

In [20]:
# 1. Base Data (Unchanged from Last Example)
X = [1000.0, 1500.0, 500.0]

pi_matrix = [
    0.70  0.20  0.10;
    0.15  0.80  0.05;
    0.25  0.15  0.60
]

@assert all(isapprox.(sum(pi_matrix, dims=2), 1.0))

# 2. DEK (2007) Parameters
α = 0.188 
β = 0.312
N = 3

# 3. Solve for Gross Manufacturing Production (Y_M)
# We solve the system: [I - (1-β)*pi'] * Y_M = α * pi' * X
I_mat = I(N)
A = I_mat - (1 - β) * pi_matrix'
b = α * pi_matrix' * X

# Use Julia's built-in linear solver (\) 
Y_M = A \ b 

# 4. Calculate Manufacturing Spending (X_M)
X_M = α .* X .+ (1 - β) .* Y_M

# 5. Calculate Total GDP (Y)
# Y = (Value Added in Mfg) + (Non-Mfg Final Demand)
Y_NM = (1 - α) .* X
Y = β .* Y_M .+ Y_NM

# 6. Calculate Deficits
D = X .- Y
D_M = X_M .- Y_M

# 7. Output the Results
println("--- 2-Sector Initial Equilibrium ---")
for i in 1:N
    println("Country $i:")
    println("  Total Spending (X) : ", round(X[i], digits=2))
    println("  Total GDP (Y)      : ", round(Y[i], digits=2))
    println("  Total Deficit (D)  : ", round(D[i], digits=2))
    println("  -------------------")
    println("  Manu. Spending (X_M) : ", round(X_M[i], digits=2))
    println("  Manu. Prod. (Y_M)    : ", round(Y_M[i], digits=2))
    println("  Manu. Deficit (D_M)  : ", round(D_M[i], digits=2))
    println()
end

# Verify the math
println("Verification:")
println("Global Total Deficit matches Manu. Deficit? ", all(isapprox.(D, D_M, atol=1e-8)))
println("Sum of all deficits = 0? ", isapprox(sum(D), 0.0, atol=1e-8))

--- 2-Sector Initial Equilibrium ---
Country 1:
  Total Spending (X) : 1000.0
  Total GDP (Y)      : 1014.37
  Total Deficit (D)  : -14.37
  -------------------
  Manu. Spending (X_M) : 634.26
  Manu. Prod. (Y_M)    : 648.63
  Manu. Deficit (D_M)  : -14.37

Country 2:
  Total Spending (X) : 1500.0
  Total GDP (Y)      : 1492.39
  Total Deficit (D)  : 7.61
  -------------------
  Manu. Spending (X_M) : 887.07
  Manu. Prod. (Y_M)    : 879.46
  Manu. Deficit (D_M)  : 7.61

Country 3:
  Total Spending (X) : 500.0
  Total GDP (Y)      : 493.23
  Total Deficit (D)  : 6.77
  -------------------
  Manu. Spending (X_M) : 286.36
  Manu. Prod. (Y_M)    : 279.6
  Manu. Deficit (D_M)  : 6.77

Verification:
Global Total Deficit matches Manu. Deficit? true
Sum of all deficits = 0? true


In [21]:
Y

3-element Vector{Float64}:
 1014.3734240397579
 1492.3922680661499
  493.2343078940921

Our GDP vector changed because we are now calculating it properly using equation 2. Similarly the deficit value changed.

Now using the solver from before under the same counterfactual situation

Manufacturing deficit = 0 

spending deficit = 0 

In [ ]:
#D_Mprime = D_M + CA #This isn't necessarily = 0 in real life/in real 2 sector model. imbalances in manufacturing and other spending can add up to be non 0
#however very conveniently in our simulated data we have set it up so that the non-manufacturing sector balances out the manufacturing sector. 
D_Mprime = zeros(3)
D_prime = zeros(3)


+ (generic function with 1 method)

In [ ]:
#Iteration parameters 
tol = 1e-6
max_iter = 10000
step_size = 0.1

#Initial guesses for endogenous variables w_hat and p_hat
w_hat = ones(3)
p_hat = ones(3)

iteration_count = 0
error = 1.

N = 3

while error > tol && iteration_count < max_iter

    #Inner Loop: Solve for prices (equation 7)
    p_error = 1
    while p_error > tol
        p_hat_update = zeros(N) #update guess of p_hat by filling in equation 7
        #need to calculate p_n for each N 
        for n in 1:N
            #Calculate term inside parantheses in equation 7
            term = sum([pi_matrix[n,k] * w_hat[k]^(-θ*β) * p_hat[k]^(-θ*(1-β)) for k in 1:N])
            p_hat_update[n] = term^(-1/θ)
        end
       
        p_error = maximum(abs.(p_hat_update - p_hat))
        p_hat = p_hat_update #norm(p_hat_update - p_hat) apparently this is better condition to guarantee convergence
    end

    #In the Outer Loop, solve Equation 6 
    #Equation 6 is a system of N equations. 1 equation for each country 
    LHS = zeros(N) #LHS is income/supply.
    RHS = zeros(N) #RHS is demand 

    for i in 1:N 
        #Calculate LHS 
        LHS[i] = w_hat[i]*Y[i] + D_prime[i] - (1/α)*D_Mprime[i] 

        #Calculate RHS
        sum_RHS = 0 
        for n in 1:N
            numerator = pi_matrix[n,i] * w_hat[i]^(-θ*β) * p_hat[i]^(-θ*(1-β))
            denominator = p_hat[n]^(-θ) #note the denominator is just an expression in terms of p_hat which we already have
            multiplicative_term = w_hat[n]*Y[n] + D_prime[n] - ((1-β)/α)*D_Mprime[n] 

            sum_RHS += (numerator/denominator)*multiplicative_term
        end
        RHS[i] = sum_RHS

    end

    #Now update wage. 
    w_hat_update = zeros(N) #update guess of w_hat 
    for i in 1:N
        #RHS (demand) > LHS (supply/income), increase wages so that LHS increases
        w_hat_update[i] = w_hat[i] * ((RHS[i]/LHS[i])^step_size)
    end

    #Need to keep GDP (Y) constant 
    current_Y = sum(w_hat_update .* Y)
    initial_Y = sum(Y)
    norm_factor = initial_Y/current_Y

    w_hat = w_hat_update*norm_factor

    #check for convergence
    error = maximum(abs.((RHS .- LHS) ./ LHS)) #norm(RHS - LHS) 
    iteration_count += 1
end

println("Convergence achieved in ", iteration_count, " iterations.")
println("Final w_hat:", w_hat)
println("Final p_hat:", p_hat)

Convergence achieved in 27 iterations.
Final w_hat:[1.0121532741892614, 0.9957857678324376, 0.9877569934749038]
Final p_hat:[1.0031508725252085, 0.9989681153484691, 0.997928914853309]


In [23]:
Y_prime = w_hat .* Y

welfare_hat = [
    (w_hat[i]/p_hat[i])^α * (1+D_prime[i]/Y_prime[i])/(1+D[i]/Y[i]) for i in 1:N
    ]

println("Counterfactual change in welfare is ", welfare_hat)

Counterfactual change in welfare is [1.0160786076806725, 0.994331544738473, 0.9845703843040046]
